In [ ]:
#betweenness centrality

from typing import NamedTuple

class User(NamedTuple):
    id: int
    name: str

users = [User(0, "Hero"), User(1, "Dunn"), User(2, "Sue"), User(3, "Chi"),
         User(4, "Thor"), User(5, "Clive"), User(6, "Hicks"),
         User(7, "Devin"), User(8, "Kate"), User(9, "Klein")]

friend_pairs = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
                (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)]

In [ ]:
from typing import Dict, List

Friendships = Dict[int, List[int]]
unique_friends = list({item for pair in friend_pairs for item in pair})
friendships: Friendships = {item: [] for item in unique_friends}

for pair in friend_pairs:
    friendships[pair[0]].append(pair[1])
    friendships[pair[1]].append(pair[0])


assert friendships[4] == [3, 5]
assert friendships[8] == [6, 7, 9]


In [44]:
from collections import deque
from typing import Dict, List



def bfs_find_all_shortest_paths(friendships: Friendships, start: int, target: int) -> List[List[int]]:
    if start == target:
        return [[start]]
    
    distances = {start: 0}
    queue = deque([[start]])
    
    all_paths = []
    min_target_length = None

    while queue:
        path = queue.popleft()
        
        if min_target_length is not None and len(path) > min_target_length:
            break    
        last_node = path[-1]
        
        for neighbor in friendships[last_node]:
            neighbor_dist = len(path)  
            
            if neighbor == target:
                if min_target_length is None:
                    min_target_length = len(path)
                if len(path) == min_target_length:
                    all_paths.append(path + [neighbor])
                continue
            
            if neighbor not in distances or distances[neighbor] == neighbor_dist:
                distances[neighbor] = neighbor_dist
                queue.append(path + [neighbor])
                
    return all_paths
bfs_find_all_shortest_paths(friendships, 0, 3)

[[0, 1, 3], [0, 2, 3]]

In [56]:
import cProfile 
import pstats

betweenness = {node: 0 for node in friendships}
nodes = list(friendships.keys())

for idx, u in enumerate(nodes):
    for v in nodes[idx + 1:]:
        paths = bfs_find_all_shortest_paths(friendships, u, v)
        num_paths = len(paths)
        
        if num_paths == 0:
            continue 
    
        for path in paths:
            for intermediary in path[1:-1]:
                betweenness[intermediary] += 1.0 / num_paths
print(betweenness)

{0: 0, 1: 3.5, 2: 3.5, 3: 18.0, 4: 20.0, 5: 20.5, 6: 6.0, 7: 6.0, 8: 8.5, 9: 0}


In [57]:
# implementacja z biblioteka networkx
import networkx as nx

friend_pairs = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
                (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)]

G = nx.Graph()
G.add_edges_from(friend_pairs)


betweenness_norm = nx.betweenness_centrality(G, normalized=True) # znormalizowane jako % wszystich paths 
betweenness_row = nx.betweenness_centrality(G, normalized=False)

for node, score in sorted(betweenness_norm.items()):
    print(f"Węzeł {node}: {score:.2f},  :{betweenness_row[node]}")

Węzeł 0: 0.00,  :0.0
Węzeł 1: 0.10,  :3.5
Węzeł 2: 0.10,  :3.5
Węzeł 3: 0.50,  :18.0
Węzeł 4: 0.56,  :20.0
Węzeł 5: 0.57,  :20.5
Węzeł 6: 0.17,  :6.0
Węzeł 7: 0.17,  :6.0
Węzeł 8: 0.24,  :8.5
Węzeł 9: 0.00,  :0.0


In [60]:
# eigenvector centrality
import numpy as np
def entry_fn(i: int, j: int):
    return 1 if (i, j) in friend_pairs or (j, i) in friend_pairs else 0

n = len(users)
adjacency_matrix = np.zeros((n, n), dtype=int)


for i, j in friend_pairs:
    adjacency_matrix[i, j] = 1
    adjacency_matrix[j, i] = 1

adjacency_matrix  

array([[0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 1, 1, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 1, 1, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 1, 1, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0]])

In [73]:
eigenvalues, eigenvectors = np.linalg.eig(adjacency_matrix)
dominant_index = np.argmax(eigenvalues.real)

centralities = eigenvectors[:, dominant_index].real

if np.sum(centralities) < 0:
    centralities = -centralities

centralities = centralities / np.sum(centralities)

user_centralities = [
    (users[i]["id"] if isinstance(users[i], dict) else users[i], score)
    for i, score in enumerate(centralities)
]

user_centralities.sort(key=lambda item: item[1], reverse=True)

print(f"{'User ID'} | {'Eigenvector Centrality'}")
for user_id, score in user_centralities:
    print(f"{user_id} | {score:.4f}")

User ID | Eigenvector Centrality
User(id=2, name='Sue') | 0.2027
User(id=1, name='Dunn') | 0.2027
User(id=3, name='Chi') | 0.1864
User(id=0, name='Hero') | 0.1519
User(id=4, name='Thor') | 0.0920
User(id=5, name='Clive') | 0.0591
User(id=6, name='Hicks') | 0.0329
User(id=7, name='Devin') | 0.0329
User(id=8, name='Kate') | 0.0287
User(id=9, name='Klein') | 0.0107


In [76]:
# PageRank
endorsements = [(0, 1), (1, 0), (0, 2), (2, 0), (1, 2),
                (2, 1), (1, 3), (2, 3), (3, 4), (5, 4),
                (5, 6), (7, 5), (6, 8), (8, 7), (8, 9)]

from collections import Counter

endorsement_counts = Counter(target for source, target in endorsements)
endorsement_counts


Counter({1: 2, 0: 2, 2: 2, 3: 2, 4: 2, 6: 1, 5: 1, 8: 1, 7: 1, 9: 1})

In [79]:
from typing import Tuple
import tqdm
def page_rank(users: List[User],
              endorsements: List[Tuple[int, int]],
              damping: float = 0.85,
              num_iters: int = 100) -> Dict[int, float]:
    # Compute how many people each person endorses
    outgoing_counts = Counter(target for source, target in endorsements)

    # Initially distribute PageRank evenly
    num_users = len(users)
    pr = {user.id : 1 / num_users for user in users}

    # Small fraction of PageRank that each node gets each iteration
    base_pr = (1 - damping) / num_users

    for iter in tqdm.trange(num_iters):
        next_pr = {user.id : base_pr for user in users}  # start with base_pr

        for source, target in endorsements:
            # Add damped fraction of source pr to target
            next_pr[target] += damping * pr[source] / outgoing_counts[source]

        pr = next_pr

    return pr

In [80]:
pr = page_rank(users, endorsements)

# Thor (user_id 4) has higher page rank than anyone else
assert pr[4] > max(page_rank
                   for user_id, page_rank in pr.items()
                   if user_id != 4)

100%|██████████| 100/100 [00:00<00:00, 418175.87it/s]
